In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mutual_info_score
from sklearn.feature_selection import mutual_info_classif

In [3]:
df = pd.read_csv('../data/diabetes_prediction_dataset_cleaned_v2.csv')

In [4]:
numerical = ['bmi', 'age', 'blood_glucose_level','HbA1c_level']
categorical = ['gender', 'hypertension', 'heart_disease', 'smoking_history']
features = ['gender','age','hypertension','heart_disease','smoking_history','bmi','HbA1c_level','blood_glucose_level']

In [5]:
#Split dataset  Proportions: Train: 60%, Validation: 20%, Test: 20%
df_full_train , df_test = train_test_split(df, test_size=0.20, random_state=42)
df_train , df_val = train_test_split(df_full_train, test_size=0.25, random_state=42)

In [6]:
#Global rate risk of diabetes
global_diabetes = df_full_train['diabetes'].mean()
print(global_diabetes)

0.0879010868999948


In [7]:
from IPython.display import display
for c in categorical:
    print(c)
    df_group = df_full_train.groupby(c).diabetes.agg(['mean', 'count'])
    df_group['diff'] = df_group['mean'] - global_diabetes
    df_group['risk'] = df_group['mean'] / global_diabetes
    display(df_group)
    print()
    print()

gender


,mean,count,diff,risk
gender,,,,
Female,0.078836,44967,-0.009065,0.896867
Male,0.100711,31933,0.012810,1.145729
Other,0.000000,16,-0.087901,0.000000




hypertension


,mean,count,diff,risk
hypertension,,,,
0,0.071857,70988,-0.016044,0.817478
1,0.280027,5928,0.192126,3.185706




heart_disease


,mean,count,diff,risk
heart_disease,,,,
0,0.077873,73800,-0.010028,0.885912
1,0.325417,3116,0.237516,3.702084




smoking_history


,mean,count,diff,risk
smoking_history,,,,
No Info,0.044434,26376,-0.043467,0.505504
current,0.103785,7371,0.015884,1.180703
ever,0.113798,3225,0.025897,1.294619
former,0.171014,7438,0.083113,1.945524
never,0.096681,27451,0.008780,1.099888
not current,0.105045,5055,0.017143,1.195031


In [ ]:
for col in numerical:
    print(col)
    # Group into 4 quartiles of equal size
    rangos = pd.qcut(df_full_train[col], q=4, duplicates='drop')

    # table risk
    df_risk = df_full_train.groupby(rangos, observed=False)['diabetes'].agg(
        ['mean', 'count']
    )
    df_risk['diff'] = df_risk['mean'] - global_diabetes
    df_risk['risk'] = df_risk['mean'] / global_diabetes

    display(df_risk)
    print()
    print()
    

bmi


,mean,count,diff,risk
bmi,,,,
"(10.009, 23.38]",0.022817,19240,-0.065084,0.259576
"(23.38, 27.32]",0.067839,30764,-0.020062,0.771766
"(27.32, 29.84]",0.102614,7689,0.014713,1.167382
"(29.84, 95.69]",0.179264,19223,0.091363,2.039388




age


,mean,count,diff,risk
age,,,,
"(0.079, 24.0]",0.006320,19938,-0.081581,0.071894
"(24.0, 43.0]",0.035533,19278,-0.052368,0.404235
"(43.0, 59.0]",0.109330,18531,0.021429,1.243788
"(59.0, 80.0]",0.204706,19169,0.116804,2.328817




blood_glucose_level


,mean,count,diff,risk
blood_glucose_level,,,,
"(79.999, 100.0]",0.000000,21665,-0.087901,0.000000
"(100.0, 140.0]",0.088281,17886,0.000380,1.004326
"(140.0, 159.0]",0.066696,23090,-0.021206,0.758757
"(159.0, 300.0]",0.255131,14275,0.167230,2.902482




HbA1c_level


,mean,count,diff,risk
HbA1c_level,,,,
"(3.499, 4.8]",0.000000,23303,-0.087901,0.000000
"(4.8, 5.8]",0.057826,18573,-0.030075,0.657852
"(5.8, 6.2]",0.082219,18925,-0.005682,0.935361
"(6.2, 9.0]",0.256345,16115,0.168444,2.916290


* The diabetes rates for both females and males are very close to the global average of the dataset (which is approximately 8.79% or 0.087901) and risk <2.0. It means A patient's gender shows no predictive influence on diabetes status and  neither smoking history does. 
* In the 'other' category We have 16 recors (all having zero diabetes)

# Mutual information (categorical variables)

In [8]:
def mutual_info_diabetes_score(series):
    return mutual_info_score(series, df_full_train.diabetes)

In [9]:
mi_cat = df_full_train[categorical].apply(mutual_info_diabetes_score)

# Mutual information for numerical variables

In [10]:
mi_numerical = mutual_info_classif(df_full_train[numerical], df_full_train.diabetes, discrete_features=False,random_state=1)

In [ ]:
mi_num = pd.Series(mi_numerical,index=numerical)
mi_num

bmi                    0.025804
age                    0.042220
blood_glucose_level    0.114823
HbA1c_level            0.133650
dtype: float64

In [14]:
mi_cat

gender             0.000736
hypertension       0.013453
heart_disease      0.009635
smoking_history    0.008793
dtype: float64

In [17]:
mutual_inf = pd.concat([mi_num,mi_cat])
mutual_inf.sort_values(ascending=False)

HbA1c_level            0.133650
blood_glucose_level    0.114823
age                    0.042220
bmi                    0.025804
hypertension           0.013453
heart_disease          0.009635
smoking_history        0.008793
gender                 0.000736
dtype: float64